In [ ]:
import ee, json, time

In [ ]:
# earthengine authenticate
ee.Authenticate()


Successfully saved authorization token.


In [ ]:
GEOJSON_PATH       = '../dubai_regions.geojson' # check the path, repo v1 gives in the parent folder
PROJECT_ID         = 'USE-YOUR-PROJECT-ID-HERE' # <-- CHANGE THIS , obtain from Google Cloud Console
START_DATE         = '2015-04-01'
END_DATE           = '2025-04-01'
CLOUD_THRESHOLD    = 20
SLEEP_BETWEEN_TASK = 2

ee.Initialize(project=PROJECT_ID)

def mask_s2(img):
    qa = img.select('QA60') # Sentinel-2 SR
    cloud  = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return img.updateMask(mask)

def add_ndbi(img):
    # B11: SWIR1, 20 m
    # B8: NIR, 10 m
    ndbi = img.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return ndbi.copyProperties(img, ['system:time_start'])

def weekly_dates(start, end):
    s = ee.Date(start)
    e = ee.Date(end)
    n_weeks = e.difference(s, 'week').subtract(1)
    return ee.List.sequence(0, n_weeks).map(lambda k: s.advance(k, 'week'))

with open(GEOJSON_PATH) as f:
    gj = json.load(f)

features = [ee.Feature(ee.Geometry(feat['geometry']),
                       {'name': feat['properties']['name']}) for feat in gj['features']]
region_fc = ee.FeatureCollection(features)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterDate(START_DATE, END_DATE)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))
      .filterBounds(region_fc.geometry())
      .map(mask_s2)
      .map(add_ndbi))

week_starts = weekly_dates(START_DATE, END_DATE)

for region_feat in features:
    reg_name = region_feat.get('name').getInfo()
    reg_geom = region_feat.geometry()

    def region_series(date):
        d_start = ee.Date(date)
        d_end   = d_start.advance(1, 'week')
        comp    = s2.filterDate(d_start, d_end).median()

        band_ok = comp.bandNames().contains('NDBI')
        mean_val = ee.Algorithms.If(
            band_ok,
            comp.select('NDBI').reduceRegion(
                reducer = ee.Reducer.mean(),
                geometry= reg_geom,
                scale   = 10,
                bestEffort=True
            ).get('NDBI'),
            None
        )

        return ee.Feature(None, {
            'region':     reg_name,
            'week_start': d_start.format('YYYY-MM-dd'),
            'ndbi':       mean_val
        })

    series_fc = ee.FeatureCollection(week_starts.map(region_series))

    ee.batch.Export.table.toDrive(
        collection     = series_fc,
        description    = f'{reg_name}_weekly_ndbi',
        fileNamePrefix = f'{reg_name}_weekly_ndbi',
        fileFormat     = 'CSV'
    ).start()

    time.sleep(SLEEP_BETWEEN_TASK)
    

print('All export tasks submitted.')


All export tasks submitted.
